-- IA — Resumos Executivos --

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv(override=True)
engine = create_engine(os.getenv("DATABASE_URL"))
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("Conexao configurada.")

Conexao configurada.


Teste — 10 Proposições

In [2]:
df_teste = pd.read_sql("SELECT id, ementa FROM proposicoes LIMIT 10", engine)
print(f"{len(df_teste)} proposicoes carregadas")
df_teste.head()

10 proposicoes carregadas


,id,ementa
0,106067,Institui o dia 02 de julho como data histórica...
1,108248,"Modifica a Lei nº 8.078, de 1990, vedando a co..."
2,110593,"Altera a Lei nº 9.503, de 23 de setembro de 19..."
3,115072,"Altera os arts. 302 e 303 da Lei nº 9.503, de ..."
4,117394,"Altera a Lei nº 9.503, de 23 de setembro de 19..."


In [3]:
def gerar_resumo(ementa):
    if not ementa or str(ementa).strip() == "":
        return None
    try:
        prompt = "Resuma esta proposicao: " + str(ementa)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Voce e um analista legislativo. Resuma proposicoes em 3 linhas, em linguagem clara para um executivo de negocios. Destaque o impacto pratico."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=150,
            temperature=0.3
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Erro: {e}")
        return None

In [4]:
df_teste["resumo"] = df_teste["ementa"].apply(gerar_resumo)
print("Teste concluido.")
df_teste[["id", "ementa", "resumo"]]

Teste concluido.


,id,ementa,resumo
0,106067,Institui o dia 02 de julho como data histórica...,A proposta estabelece o dia 2 de julho como um...
1,108248,"Modifica a Lei nº 8.078, de 1990, vedando a co...",A proposta altera a Lei nº 8.078/1990 para pro...
2,110593,"Altera a Lei nº 9.503, de 23 de setembro de 19...",A proposta altera o Código de Trânsito Brasile...
3,115072,"Altera os arts. 302 e 303 da Lei nº 9.503, de ...",A proposta altera os artigos 302 e 303 do Códi...
4,117394,"Altera a Lei nº 9.503, de 23 de setembro de 19...",A proposta modifica o Código de Trânsito Brasi...
5,119037,Estabelece limite para a taxa de juros pratica...,A proposta estabelece um teto para as taxas de...
6,130138,Assegura a concessão de aposentadoria especial...,A proposta garante aposentadoria especial para...
7,145655,Altera a redação dos §§ 1º e 2º do art. 126 da...,A proposição modifica os parágrafos 1º e 2º do...
8,154320,"Acrescenta incisos aos arts. 44, 89 e 128 da L...",A proposta visa permitir que membros da Defens...
9,159273,Obriga a Caixa Econômica Federal a divulgar os...,A proposta exige que a Caixa Econômica Federal...


Criar Coluna no Banco

In [5]:
with engine.connect() as conn:
    conn.execute(text("ALTER TABLE proposicoes ADD COLUMN IF NOT EXISTS resumo TEXT"))
    conn.commit()
print("Coluna resumo criada.")

Coluna resumo criada.


Carregar Proposições sem Resumo

In [6]:
df_todas = pd.read_sql("SELECT id, ementa FROM proposicoes WHERE resumo IS NULL AND ementa IS NOT NULL", engine)
print(f"{len(df_todas)} proposicoes restantes")

758 proposicoes restantes


Gerar e Salvar Resumos

In [7]:
from IPython.display import clear_output

with engine.connect() as conn:
    for i, (_, row) in enumerate(df_todas.iterrows()):
        resumo = gerar_resumo(row["ementa"])
        conn.execute(
            text("UPDATE proposicoes SET resumo = :resumo WHERE id = :id"),
            {"resumo": resumo, "id": row["id"]}
        )
        conn.commit()
        clear_output(wait=True)
        print(f"{i+1} de {len(df_todas)} processadas")

print("Atualizacao concluida.")

758 de 758 processadas
Atualizacao concluida.
